# `MappingProxyType` — Advanced Problems with Guided Solutions

This notebook is a second, independent set of advanced exercises on `types.MappingProxyType`.

The style is deliberately tutorial-oriented. Instead of jumping directly from a question to a finished answer, each exercise is broken into small logical steps:

1. establish the scenario,
2. predict the behavior,
3. run a focused experiment,
4. interpret the result,
5. build the solution incrementally,
6. finish with design notes and best practices.

The most important idea is still:

> `MappingProxyType` creates a **read-only view** of a mapping. It does not automatically create an immutable snapshot of the entire object graph.

## Setup

We start with the imports used throughout the notebook.

In [1]:
from types import MappingProxyType
from collections import ChainMap
from collections.abc import Mapping
from copy import deepcopy
from dataclasses import dataclass
from pprint import pprint

---

# Problem 1 — Publish live application state without exposing assignment

Suppose a service owns a mutable dictionary containing runtime state.

Other parts of the application should be able to inspect the state, but they should not be allowed to write to the dictionary directly.

The first design question is:

> Do we want a snapshot or a live view?

For this problem we want a **live view**.

## Step 1 — Create the internal state

In [2]:
state = {
    'status': 'starting',
    'requests': 0,
}

## Step 2 — Publish a mapping proxy

In [3]:
state_view = MappingProxyType(state)
state_view

mappingproxy({'status': 'starting', 'requests': 0})

The proxy supports ordinary read operations.

In [4]:
print(state_view['status'])
print(state_view.get('requests'))
print(list(state_view.items()))

starting
0
[('status', 'starting'), ('requests', 0)]


## Step 3 — Verify that direct assignment is blocked

In [5]:
try:
    state_view['status'] = 'hacked'
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


The consumer cannot assign through the proxy.

But the producer still owns the original dictionary.

## Step 4 — Mutate the producer's dictionary

In [6]:
state['status'] = 'ready'
state['requests'] += 1

In [7]:
state_view

mappingproxy({'status': 'ready', 'requests': 1})

The existing proxy sees the update immediately.

That is why the word **view** matters.

The proxy blocks one mutation path, but it remains connected to the backing mapping.

## Solution pattern

When the desired contract is:

> “You may always inspect the latest state, but you may not mutate it through this public reference.”

use a persistent proxy over the internal mapping.

In [8]:
class RuntimeState:
    def __init__(self):
        self._state = {
            'status': 'starting',
            'requests': 0,
        }
        self._view = MappingProxyType(self._state)

    @property
    def view(self):
        return self._view

    def mark_ready(self):
        self._state['status'] = 'ready'

    def record_request(self):
        self._state['requests'] += 1

In [9]:
runtime = RuntimeState()
public = runtime.view

runtime.mark_ready()
runtime.record_request()
runtime.record_request()

public

mappingproxy({'status': 'ready', 'requests': 2})

## Best practice

Create the proxy once if callers should keep a stable live view.

Returning a new proxy on every property access is usually unnecessary unless there is a specific reason to do so.

---

# Problem 2 — Detect the shallow-protection trap

A mapping proxy protects mapping operations such as item assignment.

But what happens when values stored inside the mapping are mutable?

## Step 1 — Create nested mutable state

In [10]:
config = {
    'plugins': ['json', 'yaml'],
    'limits': {
        'workers': 4,
    },
}

config_view = MappingProxyType(config)

## Step 2 — Confirm top-level assignment fails

In [11]:
try:
    config_view['plugins'] = []
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


So the mapping itself is protected.

Now test the list stored under `'plugins'`.

In [12]:
config_view['plugins'].append('toml')
config_view

mappingproxy({'plugins': ['json', 'yaml', 'toml'], 'limits': {'workers': 4}})

The append succeeds.

Why?

`MappingProxyType` controls operations on the mapping. It does not recursively wrap the values contained in that mapping.

The same issue appears with nested dictionaries.

In [13]:
config_view['limits']['workers'] = 16
config_view

mappingproxy({'plugins': ['json', 'yaml', 'toml'], 'limits': {'workers': 16}})

## Solution A — Use immutable nested values

If the nested data is naturally fixed, convert it to immutable types.

In [14]:
safe_config = {
    'plugins': ('json', 'yaml'),
    'limits': MappingProxyType({'workers': 4}),
}

safe_config_view = MappingProxyType(safe_config)
safe_config_view

mappingproxy({'plugins': ('json', 'yaml'),
              'limits': mappingproxy({'workers': 4})})

Now both the outer mapping and the nested mapping are read-only, and the plugin sequence is a tuple.

In [15]:
try:
    safe_config_view['limits']['workers'] = 99
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


## Solution B — Build a reusable recursive freezer

For structured built-in containers, we can recursively convert mutable containers into immutable/read-only equivalents.

In [16]:
def deep_freeze(value):
    if isinstance(value, dict):
        frozen_items = {
            key: deep_freeze(item)
            for key, item in value.items()
        }
        return MappingProxyType(frozen_items)

    if isinstance(value, list):
        return tuple(deep_freeze(item) for item in value)

    if isinstance(value, set):
        return frozenset(deep_freeze(item) for item in value)

    if isinstance(value, tuple):
        return tuple(deep_freeze(item) for item in value)

    return value

In [17]:
frozen = deep_freeze({
    'plugins': ['json', 'yaml'],
    'limits': {'workers': 4},
    'ports': {8000, 8001},
})

frozen

mappingproxy({'plugins': ('json', 'yaml'),
              'limits': mappingproxy({'workers': 4}),
              'ports': frozenset({8000, 8001})})

In [18]:
print(type(frozen))
print(type(frozen['plugins']))
print(type(frozen['limits']))
print(type(frozen['ports']))

<class 'mappingproxy'>
<class 'tuple'>
<class 'mappingproxy'>
<class 'frozenset'>


## Important limitation

A recursive helper like this only knows how to freeze the types it explicitly handles.

A custom mutable object stored as a value can still remain mutable.

So deep immutability is a data-model design decision, not something `MappingProxyType` solves by itself.

---

# Problem 3 — Live view versus read-only snapshot

Two APIs can both return a `mappingproxy` and still have very different semantics.

Compare:

```python
MappingProxyType(data)
```

with:

```python
MappingProxyType(data.copy())
```

## Step 1 — Create both forms

In [19]:
data = {'score': 10}

live = MappingProxyType(data)
snapshot = MappingProxyType(data.copy())

At this moment, both mappings contain the same values.

In [20]:
print(live)
print(snapshot)
print(live == snapshot)

{'score': 10}
{'score': 10}
True


## Step 2 — Mutate the producer

In [21]:
data['score'] = 99

In [22]:
print('live:    ', live)
print('snapshot:', snapshot)

live:     {'score': 99}
snapshot: {'score': 10}


The difference is now obvious.

The first proxy references the original dictionary.

The second proxy references a new dictionary created by `copy()`.

## Step 3 — Recognize the shallow-copy limitation

The snapshot is only top-level independent.

In [23]:
nested = {
    'items': ['A'],
}

shallow_snapshot = MappingProxyType(nested.copy())

print(nested['items'] is shallow_snapshot['items'])

True


In [24]:
nested['items'].append('B')
print('original:', nested)
print('snapshot:', shallow_snapshot)

original: {'items': ['A', 'B']}
snapshot: {'items': ['A', 'B']}


The nested list is shared.

If the contract requires a nested structural snapshot, use `deepcopy` before publishing.

In [25]:
deep_snapshot = MappingProxyType(deepcopy(nested))

nested['items'].append('C')

print('original:     ', nested)
print('deep snapshot:', deep_snapshot)

original:      {'items': ['A', 'B', 'C']}
deep snapshot: {'items': ['A', 'B']}


## Design rule

Choose the publication strategy based on semantics:

- live read-only view → `MappingProxyType(data)`
- read-only top-level snapshot → `MappingProxyType(data.copy())`
- detached nested snapshot → often `MappingProxyType(deepcopy(data))`

Remember that `deepcopy` detaches nested objects but does not automatically make every copied value immutable.

---

# Problem 4 — Build a controlled handler registry

A framework maintains a dictionary of handlers.

Requirements:

- callers can inspect registered handlers,
- registration must go through validation,
- duplicate names are forbidden,
- callers must not assign into the registry directly.

## Step 1 — Start with the naive implementation

In [26]:
class NaiveRegistry:
    handlers = {}

    @classmethod
    def register(cls, name, handler):
        cls.handlers[name] = handler

The problem is that the public dictionary is writable.

In [27]:
NaiveRegistry.register('text', str)
NaiveRegistry.handlers['text'] = int
NaiveRegistry.handlers

{'text': int}

External code bypassed the intended API completely.

## Step 2 — Separate storage from the public view

In [28]:
class HandlerRegistry:
    _handlers = {}
    handlers = MappingProxyType(_handlers)

    @classmethod
    def register(cls, name, handler):
        if not isinstance(name, str) or not name:
            raise ValueError('name must be a non-empty string')

        if not callable(handler):
            raise TypeError('handler must be callable')

        if name in cls._handlers:
            raise ValueError(f'{name!r} is already registered')

        cls._handlers[name] = handler

## Step 3 — Register through the supported API

In [29]:
HandlerRegistry.register('text', str)
HandlerRegistry.register('integer', int)
HandlerRegistry.handlers

mappingproxy({'text': str, 'integer': int})

## Step 4 — Attempt to bypass validation

In [30]:
try:
    HandlerRegistry.handlers['bad'] = object
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


## Step 5 — Observe later legitimate changes

In [31]:
HandlerRegistry.register('float', float)
HandlerRegistry.handlers

mappingproxy({'text': str, 'integer': int, 'float': float})

The public proxy is both:

- read-only to consumers,
- live with respect to controlled internal mutation.

## Subtle pitfall — Replacing the backing dictionary

A proxy points to a mapping object, not to a variable name.

If `_handlers` is rebound to a completely new dictionary, the old proxy remains attached to the old dictionary.

In [32]:
original_storage = HandlerRegistry._handlers
original_view = HandlerRegistry.handlers

HandlerRegistry._handlers = {'replacement': len}

print('new storage:', HandlerRegistry._handlers)
print('old view:   ', original_view)

new storage: {'replacement': <built-in function len>}
old view:    {'text': <class 'str'>, 'integer': <class 'int'>, 'float': <class 'float'>}


For a stable live-view design, mutate the existing backing dictionary rather than replacing it.

Restore the original storage before continuing.

In [33]:
HandlerRegistry._handlers = original_storage

---

# Problem 5 — Use the correct abstraction: `Mapping`, not `dict`

A function only needs to read key/value pairs.

Should it require `dict`?

No. A mapping proxy is not a `dict`, even though it supports the read-only mapping interface.

## Step 1 — Compare concrete-type checks

In [34]:
regular = {'x': 1}
readonly = MappingProxyType(regular)

print(isinstance(regular, dict))
print(isinstance(readonly, dict))

True
False


A strict `dict` check rejects the proxy.

## Step 2 — Check against `Mapping`

In [35]:
print(isinstance(regular, Mapping))
print(isinstance(readonly, Mapping))

True
True


Both objects satisfy the read-only mapping abstraction.

## Step 3 — Write an interface-friendly validator

In [36]:
def require_keys(mapping, *required):
    if not isinstance(mapping, Mapping):
        raise TypeError('expected a mapping')

    missing = [key for key in required if key not in mapping]

    if missing:
        raise KeyError(f'missing keys: {missing}')

    return True

In [37]:
print(require_keys(regular, 'x'))
print(require_keys(readonly, 'x'))

True
True


## Best practice

If a function only reads from an input, type against a read-only abstraction.

Require mutation-capable types only when mutation is genuinely part of the contract.

---

# Problem 6 — Build a live feature-flag service

Feature flags are a strong real-world use case for mapping proxies.

Requirements:

- application code reads flags frequently,
- administrative code updates flags,
- application code must not change flags directly,
- existing consumers should observe updates.

## Step 1 — Design the internal representation

The service owns one mutable dictionary and one persistent proxy.

In [38]:
class FeatureFlags:
    def __init__(self, initial=None):
        self._flags = dict(initial or {})
        self._view = MappingProxyType(self._flags)

    @property
    def flags(self):
        return self._view

    def enable(self, name):
        self._flags[name] = True

    def disable(self, name):
        self._flags[name] = False

    def remove(self, name):
        self._flags.pop(name, None)

## Step 2 — Keep one consumer view

In [39]:
flags = FeatureFlags({
    'checkout_v2': False,
    'recommendations': True,
})

consumer = flags.flags
consumer

mappingproxy({'checkout_v2': False, 'recommendations': True})

## Step 3 — Change state through the service

In [40]:
flags.enable('checkout_v2')
consumer

mappingproxy({'checkout_v2': True, 'recommendations': True})

The same object still works as a live view.

## Step 4 — Block direct consumer mutation

In [41]:
try:
    consumer['recommendations'] = False
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


## Step 5 — Compare with a snapshot

In [42]:
snapshot = dict(flags.flags)
flags.disable('recommendations')

print('live:    ', consumer)
print('snapshot:', snapshot)

live:     {'checkout_v2': True, 'recommendations': False}
snapshot: {'checkout_v2': True, 'recommendations': True}


This highlights a useful API decision:

- live operational state often benefits from a proxy,
- audit/history data usually benefits from snapshots.

---

# Problem 7 — Protect registry records as well as the registry itself

A mapping proxy prevents callers from replacing values in the mapping.

But if the values themselves are mutable, callers may still change internal state.

## Step 1 — Demonstrate the leak

In [43]:
users = {
    1: {'name': 'Ada', 'role': 'admin'},
    2: {'name': 'Linus', 'role': 'developer'},
}

users_view = MappingProxyType(users)

In [44]:
users_view[1]['role'] = 'guest'
users_view

mappingproxy({1: {'name': 'Ada', 'role': 'guest'},
              2: {'name': 'Linus', 'role': 'developer'}})

The outer dictionary is protected, but the nested user record is not.

## Step 2 — Model records with a frozen dataclass

In [45]:
@dataclass(frozen=True)
class User:
    name: str
    role: str

In [46]:
safe_users = {
    1: User('Ada', 'admin'),
    2: User('Linus', 'developer'),
}

safe_users_view = MappingProxyType(safe_users)
safe_users_view

mappingproxy({1: User(name='Ada', role='admin'),
              2: User(name='Linus', role='developer')})

## Step 3 — Try to mutate a record

In [47]:
try:
    safe_users_view[1].role = 'guest'
except Exception as ex:
    print(type(ex).__name__, ex)

FrozenInstanceError cannot assign to field 'role'


Now both layers are protected by different mechanisms:

- mapping proxy → protects mapping structure,
- frozen dataclass → protects record fields.

This is often cleaner than trying to force one mechanism to solve every immutability problem.

---

# Problem 8 — Layered configuration with `ChainMap`

Applications often combine configuration from several sources.

For example:

1. runtime overrides,
2. environment settings,
3. defaults.

`ChainMap` provides a live composite lookup view.

Can a `MappingProxyType` be placed around it?

## Step 1 — Build the layers

In [48]:
defaults = {
    'host': 'localhost',
    'port': 8000,
    'debug': False,
}

environment = {
    'port': 8080,
}

runtime_overrides = {
    'debug': True,
}

## Step 2 — Build the layered mapping

In [49]:
combined = ChainMap(runtime_overrides, environment, defaults)

print(combined['host'])
print(combined['port'])
print(combined['debug'])

localhost
8080
True


The first mapping containing a key wins.

## Step 3 — Publish it read-only

In [50]:
combined_view = MappingProxyType(combined)
combined_view

mappingproxy({'host': 'localhost', 'port': 8080, 'debug': True})

## Step 4 — Block consumer assignment

In [51]:
try:
    combined_view['port'] = 9999
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


## Step 5 — Change one underlying layer

In [52]:
runtime_overrides['port'] = 9000
combined_view['port']

9000

The update passes through two live layers:

- `MappingProxyType` observes the `ChainMap`,
- `ChainMap` observes its underlying mappings.

## Step 6 — Produce a merged snapshot instead

In [53]:
merged_snapshot = MappingProxyType(dict(combined))

runtime_overrides['host'] = 'runtime-host'

print('live layered view:', combined_view['host'])
print('merged snapshot:   ', merged_snapshot['host'])

live layered view: runtime-host
merged snapshot:    localhost


The snapshot materialized the effective key/value pairs at one point in time.

That may be exactly what you want for logging, auditing, or historical records.

---

# Problem 9 — Equality does not reveal liveness

A dictionary, a proxy, and a copy can compare equal while having very different behavior.

## Step 1 — Create the objects

In [54]:
data = {'a': 1, 'b': 2}
proxy = MappingProxyType(data)
copied = proxy.copy()

## Step 2 — Compare types

In [55]:
print(type(data))
print(type(proxy))
print(type(copied))

<class 'dict'>
<class 'mappingproxy'>
<class 'dict'>


`proxy.copy()` returns a normal `dict`.

## Step 3 — Compare equality

In [56]:
print(data == proxy)
print(proxy == copied)
print(data == copied)

True
True
True


All three can compare equal by content.

But equality does not tell us whether future producer updates will be visible.

## Step 4 — Mutate the original

In [57]:
data['c'] = 3

print('data:  ', data)
print('proxy: ', proxy)
print('copied:', copied)

data:   {'a': 1, 'b': 2, 'c': 3}
proxy:  {'a': 1, 'b': 2, 'c': 3}
copied: {'a': 1, 'b': 2}


The proxy tracks the original; the copy does not.

## Lesson

Do not infer publication semantics from initial equality.

Test or document whether the object is:

- live,
- snapshot-based,
- mutable,
- read-only.

---

# Problem 10 — Understand why class `__dict__` is a mapping proxy

Python itself uses mapping proxies for class namespaces.

This gives us a concrete example of the pattern in the language runtime.

## Step 1 — Define a class

In [58]:
class Service:
    version = 1

    def run(self):
        return 'running'

## Step 2 — Inspect the namespace

In [59]:
namespace = Service.__dict__
namespace

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'version': 1,
              'run': <function __main__.Service.run(self)>,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Service' objects>,
              '__weakref__': <attribute '__weakref__' of 'Service' objects>,
              '__doc__': None})

In [60]:
type(namespace)

mappingproxy

## Step 3 — Attempt dictionary-style mutation

In [61]:
try:
    Service.__dict__['version'] = 2
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


Direct dictionary mutation is blocked.

But class attributes can still be changed through the supported attribute mechanism.

## Step 4 — Change the class attribute normally

In [62]:
Service.version = 2
namespace['version']

2

The previously obtained namespace proxy reports the updated value.

This is the same design idea we have been using:

- expose introspection,
- control the mutation mechanism.

---

# Problem 11 — Build an atomic versioned configuration publisher

Now we combine mapping proxies with validation and transactional thinking.

Requirements:

- consumers receive a live read-only view,
- successful updates increment a version number,
- invalid updates do not change data,
- invalid updates do not increment the version,
- bulk updates are all-or-nothing.

## Step 1 — Create the basic object

In [63]:
class VersionedConfig:
    def __init__(self, initial=None):
        self._data = dict(initial or {})
        self._view = MappingProxyType(self._data)
        self._version = 0

    @property
    def data(self):
        return self._view

    @property
    def version(self):
        return self._version

## Step 2 — Centralize validation

For this exercise:

- keys must be non-empty strings,
- values may not be `None`.

In [64]:
class VersionedConfig:
    def __init__(self, initial=None):
        self._data = dict(initial or {})
        self._view = MappingProxyType(self._data)
        self._version = 0

    @property
    def data(self):
        return self._view

    @property
    def version(self):
        return self._version

    @staticmethod
    def _validate_pair(key, value):
        if not isinstance(key, str) or not key:
            raise ValueError('keys must be non-empty strings')

        if value is None:
            raise ValueError('values cannot be None')

## Step 3 — Add single-key mutation

In [65]:
class VersionedConfig:
    def __init__(self, initial=None):
        self._data = dict(initial or {})
        self._view = MappingProxyType(self._data)
        self._version = 0

    @property
    def data(self):
        return self._view

    @property
    def version(self):
        return self._version

    @staticmethod
    def _validate_pair(key, value):
        if not isinstance(key, str) or not key:
            raise ValueError('keys must be non-empty strings')

        if value is None:
            raise ValueError('values cannot be None')

    def set(self, key, value):
        self._validate_pair(key, value)
        self._data[key] = value
        self._version += 1

## Step 4 — Test a successful change

In [66]:
vc = VersionedConfig({'mode': 'dev'})
view = vc.data

vc.set('mode', 'prod')

print('version:', vc.version)
print('view:   ', view)

version: 1
view:    {'mode': 'prod'}


## Step 5 — Think about a naive bulk update

This implementation is dangerous:

```python
for key, value in updates.items():
    validate(key, value)
    self._data[key] = value
```

If the third entry is invalid, the first two may already have been written.

Instead, validate every pair first and commit only after all validation succeeds.

In [67]:
class VersionedConfig:
    def __init__(self, initial=None):
        self._data = dict(initial or {})
        self._view = MappingProxyType(self._data)
        self._version = 0

    @property
    def data(self):
        return self._view

    @property
    def version(self):
        return self._version

    @staticmethod
    def _validate_pair(key, value):
        if not isinstance(key, str) or not key:
            raise ValueError('keys must be non-empty strings')

        if value is None:
            raise ValueError('values cannot be None')

    def set(self, key, value):
        self._validate_pair(key, value)
        self._data[key] = value
        self._version += 1

    def update_many(self, updates):
        proposed = dict(updates)

        for key, value in proposed.items():
            self._validate_pair(key, value)

        self._data.update(proposed)
        self._version += 1

## Step 6 — Verify successful bulk mutation

In [68]:
vc = VersionedConfig({'a': 1, 'b': 2})
public = vc.data

vc.update_many({
    'a': 10,
    'c': 30,
})

print('version:', vc.version)
print('public: ', public)

version: 1
public:  {'a': 10, 'b': 2, 'c': 30}


## Step 7 — Verify failure atomicity

In [69]:
before_data = dict(vc.data)
before_version = vc.version

try:
    vc.update_many({
        'x': 100,
        'bad': None,
        'y': 200,
    })
except ValueError as ex:
    print(type(ex).__name__, ex)

print('data unchanged:   ', dict(vc.data) == before_data)
print('version unchanged:', vc.version == before_version)

ValueError values cannot be None
data unchanged:    True
version unchanged: True


## Design lesson

The mapping proxy protects the public boundary.

Validation and atomic update logic protect internal invariants.

A proxy is an access-control tool, not a replacement for correct mutation logic.

---

# Problem 12 — Publish cache statistics safely

A cache wants to expose live statistics:

- hits,
- misses,
- current size.

Consumers should see updates but should not be able to falsify the numbers.

## Step 1 — Build the cache

In [70]:
class SimpleCache:
    def __init__(self):
        self._cache = {}
        self._stats = {
            'hits': 0,
            'misses': 0,
            'size': 0,
        }
        self._stats_view = MappingProxyType(self._stats)

    @property
    def stats(self):
        return self._stats_view

    def get(self, key, loader):
        if key in self._cache:
            self._stats['hits'] += 1
            return self._cache[key]

        self._stats['misses'] += 1
        value = loader(key)
        self._cache[key] = value
        self._stats['size'] = len(self._cache)
        return value

## Step 2 — Obtain the statistics view before doing any work

In [71]:
cache = SimpleCache()
stats = cache.stats
stats

mappingproxy({'hits': 0, 'misses': 0, 'size': 0})

## Step 3 — Cause a cache miss

In [72]:
result = cache.get('python', lambda key: key.upper())
print(result)
print(stats)

PYTHON
{'hits': 0, 'misses': 1, 'size': 1}


## Step 4 — Cause a cache hit

In [73]:
result = cache.get('python', lambda key: 'SHOULD NOT RUN')
print(result)
print(stats)

PYTHON
{'hits': 1, 'misses': 1, 'size': 1}


The same `stats` proxy reflects each internal update.

## Step 5 — Attempt to falsify the metrics

In [74]:
try:
    stats['hits'] = 1_000_000
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


This is a useful observability pattern:

> expose current state without exposing the capability to mutate it through the public interface.

---

# Problem 13 — Why read-only does not imply hashable

A common misconception is:

> “If I cannot mutate an object, it must be safe to hash.”

A mapping proxy demonstrates why that reasoning is incomplete.

## Step 1 — Create a proxy

In [75]:
data = {'a': 1}
proxy = MappingProxyType(data)

## Step 2 — Try to hash it

In [76]:
try:
    print(hash(proxy))
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError unhashable type: 'dict'


Even though callers cannot mutate the proxy directly, its visible contents can change because the underlying mapping can change.

In [77]:
print('before:', proxy)
data['b'] = 2
print('after: ', proxy)

before: {'a': 1}
after:  {'a': 1, 'b': 2}


## Conceptual lesson

These properties are different:

- read-only through one reference,
- deeply immutable,
- content-stable over time,
- hashable.

Do not treat them as synonyms.

---

# Problem 14 — Inventory API with a live read-only public state

We finish the main section with a larger design exercise.

Requirements:

- quantities are stored by product code,
- public callers receive a live read-only mapping,
- negative quantities are impossible,
- `receive(product, amount)` adds stock,
- `ship(product, amount)` removes stock,
- over-shipping fails without changing state,
- products with zero remaining quantity are removed.

## Step 1 — Write validation helpers

In [78]:
def validate_product(product):
    if not isinstance(product, str) or not product:
        raise ValueError('product must be a non-empty string')


def validate_amount(amount):
    if not isinstance(amount, int) or isinstance(amount, bool) or amount <= 0:
        raise ValueError('amount must be a positive integer')

Why explicitly reject `bool`?

Because `bool` is a subclass of `int` in Python.

In [79]:
print(isinstance(True, int))

True


For inventory quantities, accepting `True` as the number `1` would be surprising.

## Step 2 — Build the object and public view

In [80]:
class Inventory:
    def __init__(self):
        self._quantities = {}
        self._view = MappingProxyType(self._quantities)

    @property
    def quantities(self):
        return self._view

## Step 3 — Implement receiving stock

In [81]:
class Inventory:
    def __init__(self):
        self._quantities = {}
        self._view = MappingProxyType(self._quantities)

    @property
    def quantities(self):
        return self._view

    def receive(self, product, amount):
        validate_product(product)
        validate_amount(amount)

        self._quantities[product] = (
            self._quantities.get(product, 0) + amount
        )

## Step 4 — Implement shipping

Validate before mutation so a failed shipment leaves the dictionary unchanged.

In [82]:
class Inventory:
    def __init__(self):
        self._quantities = {}
        self._view = MappingProxyType(self._quantities)

    @property
    def quantities(self):
        return self._view

    def receive(self, product, amount):
        validate_product(product)
        validate_amount(amount)

        self._quantities[product] = (
            self._quantities.get(product, 0) + amount
        )

    def ship(self, product, amount):
        validate_product(product)
        validate_amount(amount)

        available = self._quantities.get(product, 0)

        if amount > available:
            raise ValueError(
                f'cannot ship {amount}; only {available} available'
            )

        remaining = available - amount

        if remaining == 0:
            del self._quantities[product]
        else:
            self._quantities[product] = remaining

## Step 5 — Exercise the normal workflow

In [83]:
inventory = Inventory()
public_inventory = inventory.quantities

inventory.receive('BOOK', 10)
inventory.receive('PEN', 25)

print(public_inventory)

inventory.ship('BOOK', 3)
print(public_inventory)

{'BOOK': 10, 'PEN': 25}
{'BOOK': 7, 'PEN': 25}


## Step 6 — Remove a product when quantity reaches zero

In [84]:
inventory.ship('BOOK', 7)
public_inventory

mappingproxy({'PEN': 25})

## Step 7 — Verify failed shipping is atomic

In [85]:
before = dict(public_inventory)

try:
    inventory.ship('PEN', 100)
except ValueError as ex:
    print(type(ex).__name__, ex)

after = dict(public_inventory)

print('unchanged:', before == after)
print('state:    ', public_inventory)

ValueError cannot ship 100; only 25 available
unchanged: True
state:     {'PEN': 25}


## Step 8 — Verify callers cannot fake inventory

In [86]:
try:
    public_inventory['PEN'] = 999_999
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError 'mappingproxy' object does not support item assignment


## Final solution analysis

The design separates responsibilities cleanly:

- `MappingProxyType` prevents public dictionary assignment,
- validation prevents invalid inputs,
- `ship()` enforces the stock invariant,
- mutation happens only through controlled methods.

That is a much stronger design than merely wrapping a dictionary and assuming the problem is solved.

---

# Bonus Problem A — Historical snapshots

Suppose every configuration commit must preserve history.

A live proxy is the wrong tool for each historical entry because old history must not change when current state changes.

## Step 1 — Build the history manager

In [87]:
class ConfigHistory:
    def __init__(self, initial=None):
        self._current = dict(initial or {})
        self._history = []

    @property
    def current(self):
        return MappingProxyType(self._current)

    @property
    def history(self):
        return tuple(self._history)

    def commit(self, **changes):
        self._current.update(changes)
        snapshot = MappingProxyType(deepcopy(self._current))
        self._history.append(snapshot)
        return snapshot

## Step 2 — Create several versions

In [88]:
history = ConfigHistory({
    'theme': 'light',
    'roles': ['user'],
})

v1 = history.commit()
v2 = history.commit(theme='dark')

print('v1:', v1)
print('v2:', v2)

v1: {'theme': 'light', 'roles': ['user']}
v2: {'theme': 'dark', 'roles': ['user']}


## Step 3 — Change current nested state

For demonstration, we mutate the producer's current state directly.

In [89]:
history._current['roles'].append('admin')

print('current:', history.current)
print('v1:     ', v1)
print('v2:     ', v2)

current: {'theme': 'dark', 'roles': ['user', 'admin']}
v1:      {'theme': 'light', 'roles': ['user']}
v2:      {'theme': 'dark', 'roles': ['user']}


Because each historical snapshot used `deepcopy`, old versions remain detached from later nested mutations.

This is a case where snapshot semantics are essential.

---

# Bonus Problem B — Compare four publication strategies

We can summarize much of the notebook by comparing four common patterns.

In [90]:
source = {
    'count': 1,
    'nested': [],
}

original = source
mutable_copy = source.copy()
live_readonly = MappingProxyType(source)
readonly_snapshot = MappingProxyType(source.copy())

## Step 1 — Mutate the producer at the top level

In [91]:
source['count'] = 2

print('original:         ', original['count'])
print('mutable copy:     ', mutable_copy['count'])
print('live readonly:    ', live_readonly['count'])
print('readonly snapshot:', readonly_snapshot['count'])

original:          2
mutable copy:      1
live readonly:     2
readonly snapshot: 1


## Step 2 — Mutate a nested shared value

In [92]:
source['nested'].append('X')

print('original:         ', original['nested'])
print('mutable copy:     ', mutable_copy['nested'])
print('live readonly:    ', live_readonly['nested'])
print('readonly snapshot:', readonly_snapshot['nested'])

original:          ['X']
mutable copy:      ['X']
live readonly:     ['X']
readonly snapshot: ['X']


The shallow copies still share the nested list.

This experiment is useful because it separates two axes:

1. whether the top-level mapping is shared,
2. whether nested objects are shared.

## Summary table

| Strategy | Public assignment? | Sees producer top-level updates? | Shallow nested values shared? |
|---|---:|---:|---:|
| original dict | yes | yes | yes |
| `dict.copy()` | yes | no | yes |
| `MappingProxyType(data)` | no | yes | yes |
| `MappingProxyType(data.copy())` | no | no | yes |

For nested independence, consider `deepcopy` or immutable nested value types.

---

# Bonus Problem C — Explicit helper names for clear semantics

A vague helper such as `readonly_mapping()` leaves an important question unanswered:

> Is the result live or snapshot-based?

Prefer names that encode the contract.

In [93]:
def readonly_view(mapping):
    if not isinstance(mapping, Mapping):
        raise TypeError('expected a mapping')

    return MappingProxyType(mapping)


def readonly_snapshot(mapping):
    if not isinstance(mapping, Mapping):
        raise TypeError('expected a mapping')

    return MappingProxyType(dict(mapping))

## Step 1 — Compare their behavior

In [94]:
source = {'score': 10}

live = readonly_view(source)
snapshot = readonly_snapshot(source)

source['score'] = 20

print('live:    ', live)
print('snapshot:', snapshot)

live:     {'score': 20}
snapshot: {'score': 10}


The function names communicate the key semantic difference before the caller even reads the implementation.

Good naming is part of API correctness.

---

# Bonus Problem D — Test the complete “live read-only” contract

A good test should verify both halves of the promise.

If documentation says “live read-only view,” then tests should prove:

- reads work,
- direct writes fail,
- producer updates are visible later.

In [95]:
def test_live_readonly_contract():
    source = {'x': 1}
    view = MappingProxyType(source)

    # Reading works.
    assert view['x'] == 1

    # Consumer assignment fails.
    try:
        view['x'] = 999
    except TypeError:
        pass
    else:
        raise AssertionError('proxy unexpectedly allowed assignment')

    # Producer changes are visible.
    source['x'] = 2
    source['y'] = 3

    assert view['x'] == 2
    assert view['y'] == 3

    return 'all contract checks passed'

In [96]:
test_live_readonly_contract()

'all contract checks passed'

Testing only the `TypeError` would verify “read-only” but not “live.”

Testing only producer updates would verify “live” but not “read-only.”

Contract-oriented tests should cover both.

---

# Short Prediction Drills

Try to answer each question before running the corresponding code cell.

## Drill 1

What does this print?

```python
d = {'x': 1}
p = MappingProxyType(d)
d.clear()
print(len(p))
```

In [97]:
d = {'x': 1}
p = MappingProxyType(d)
d.clear()
print(len(p))

0


**Solution:** `0`.

The proxy is live, so it immediately observes that the backing dictionary is empty.

## Drill 2

Does this mutation succeed?

```python
d = {'numbers': [1, 2]}
p = MappingProxyType(d)
p['numbers'][0] = 999
```

In [98]:
d = {'numbers': [1, 2]}
p = MappingProxyType(d)
p['numbers'][0] = 999
p

mappingproxy({'numbers': [999, 2]})

**Solution:** yes.

The list is mutable. The proxy only protects the mapping operation.

## Drill 3

What type does `proxy.copy()` return?

In [99]:
proxy = MappingProxyType({'a': 1})
print(type(proxy.copy()))

<class 'dict'>


**Solution:** `dict`.

If you need the copy itself to be read-only, wrap that copied dictionary in another `MappingProxyType`.

## Drill 4

Which is live?

```python
A = MappingProxyType(d)
B = MappingProxyType(d.copy())
```

**Solution:** `A` is live with respect to `d`.

`B` wraps a separate copied dictionary, so later top-level changes to `d` are not reflected in `B`.

## Drill 5

Why is this class only partially protected?

```python
class Store:
    _data = {'users': []}
    data = MappingProxyType(_data)
```

In [100]:
class Store:
    _data = {'users': []}
    data = MappingProxyType(_data)

Store.data['users'].append('Ada')
Store.data

mappingproxy({'users': ['Ada']})

**Solution:** the mapping is protected, but the list stored inside it is mutable.

The class needs immutable nested values, frozen records, copies, or another policy depending on its intended contract.

---

# Best Practices

## 1. Document whether the result is live or snapshot-based

“Read-only mapping” is often not enough.

Prefer precise descriptions such as:

- “live read-only view of the registry,”
- “read-only snapshot of configuration at call time.”

## 2. Remember that the protection is shallow

A proxy does not recursively freeze lists, dictionaries, sets, or custom mutable objects stored as values.

## 3. Keep legitimate mutation behind methods

Use validated methods when mutation must preserve invariants.

## 4. Prefer `Mapping` for read-only input contracts

Do not require a concrete `dict` when any mapping implementation is acceptable.

## 5. Do not accidentally rebind the backing mapping

A proxy points to a particular object. Replacing the internal variable does not retarget an existing proxy.

## 6. Separate detachment from immutability

A deep copy gives nested independence.

Immutable value types prevent mutation.

Those are related but different guarantees.

## 7. Test the semantics you actually promise

For a live read-only API, test both:

- direct mutation is blocked,
- internal updates remain visible.

---

# Final Review

`MappingProxyType` is best understood as a controlled access mechanism.

It allows consumers to:

- inspect keys,
- retrieve values,
- iterate,
- use normal read-only mapping operations.

It prevents consumers from assigning or deleting mapping entries through that proxy.

It does **not** automatically provide:

- deep immutability,
- snapshot semantics,
- hashability,
- independence from nested mutable values,
- validation,
- thread safety.

The central design question is always:

> What exact mutation and update semantics should the public API expose?

Once that is clear, `MappingProxyType` becomes a precise and powerful tool rather than simply a “read-only dictionary.”